In [1]:
import os, sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "src" / "pipeline" / "config.py").is_file():
    if _p.parent == _p:
        raise SystemExit("please run this notebook from inside the repository")
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

FOLD = 0  # 0-4, or None = all five folds

import numpy as np
import torch
from pipeline import config
from pipeline.test import casee_eval as ce
DEV = ce.device()
FOLDS = range(5) if FOLD is None else [FOLD]


In [2]:
import pickle as pk

from pipeline.test.casee_eval import predict_mut_table

EXT_IDX = 0   # dataset calibration applied zero-shot
COMBOS = ["Q695A", "Q695A:Q926A", "Q926A", "R661A:Q695A", "R661A:Q695A:Q926A"]

md = pk.load(open(config.HF1_DIR / "mut_data_0_0_0.pkl", "rb"))
sites = pk.load(open(config.SIM_GETDATA / "sites_data.pkl", "rb"))

on = {}
for k in FOLDS:
    net, res = ce.load_sim(k, DEV), ce.load_residual(k, DEV)
    rows = np.load(config.HF1_DIR / ("hf1_fold%d.npy" % k), allow_pickle=True)
    y, z = predict_mut_table(net, res, rows, md, sites, DEV, idx=EXT_IDX)
    on[k] = ce.rho(y, z)
    print("fold %d  Wang HF1 on-target  n=%d  Spearman = %.4f" % (k, len(y), on[k]))
v = list(on.values())
if len(v) > 1:
    print("Wang HF1 on-target  mean = %.4f  std = %.4f  (folds=%d)" % (np.mean(v), np.std(v), len(v)))


fold 0  Wang HF1 on-target  n=10788  Spearman = 0.6203


In [3]:
print()
for k in FOLDS:
    out = ce.variant_groups(k, DEV)
    print("--- fold %d  off-target, Cas9 variants" % k)
    for c in COMBOS:
        d = out["Kleinstiver " + c]
        yt = np.asarray(d["y_true"], float) / 100.0
        yp = np.asarray(d["y_pred"], float)
        print("%-22s n=%d  r = %.3f  R = %.3f" % (c, len(yt), ce.rho(yt, yp, "pearson"), ce.rho(yt, yp)))



--- fold 0  off-target, Cas9 variants
Q695A                  n=5  r = 0.636  R = 0.700
Q695A:Q926A            n=5  r = 0.954  R = 0.700
Q926A                  n=5  r = 0.497  R = 0.400
R661A:Q695A            n=5  r = 0.928  R = 0.900
R661A:Q695A:Q926A      n=5  r = 0.963  R = 0.900
